# P2-2 · Bài 2 · Train mạng thật

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
# CIFAR-10 thật (Colab, cần mạng): torchvision.datasets.CIFAR10("data", train=True, download=True, transform=T.ToTensor())
# ở đây dùng FakeData: ảnh ngẫu nhiên cùng định dạng, không cần tải
train_ds = torchvision.datasets.FakeData(size=2000, image_size=(3, 32, 32), num_classes=10, transform=T.ToTensor())
val_ds = torchvision.datasets.FakeData(size=500, image_size=(3, 32, 32), num_classes=10, transform=T.ToTensor(), random_offset=1)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
for x, y in train_loader:      # mỗi vòng lặp trả về một batch [128, 3, 32, 32] và [128]
    print(x.shape, y.shape, len(train_loader), "batch mỗi epoch")
    break

torch.Size([128, 3, 32, 32]) torch.Size([128]) 16 batch mỗi epoch


In [ ]:
import torch.nn as nn
model, epochs = nn.Linear(10, 2), 30          # model nhỏ chỉ để xem lịch learning rate
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=len(train_loader) * epochs, pct_start=0.05)
lrs = []
for step in range(len(train_loader) * epochs):
    opt.step()                                # trong vòng lặp thật: backward() rồi opt.step()
    sched.step()                              # ... và gọi sched.step() ngay sau opt.step()
    lrs.append(sched.get_last_lr()[0])
peak = max(lrs)
print(f"{len(lrs)} bước: lr bắt đầu {lrs[0]:.1e}, đạt đỉnh {peak:.1e} ở bước {lrs.index(peak)}, kết thúc {lrs[-1]:.1e}")

480 bước: lr bắt đầu 1.3e-05, đạt đỉnh 3.0e-04 ở bước 22, kết thúc 4.8e-09


In [ ]:
import torch, torch.nn as nn
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.MaxPool2d(2))
model = nn.Sequential(block(3, 32), block(32, 64), block(64, 128),     # 32→16→8→4
                      nn.Flatten(), nn.Linear(128 * 4 * 4, 10))
print(model(torch.randn(8, 3, 32, 32)).shape, sum(p.numel() for p in model.parameters()), "tham số")

torch.Size([8, 10]) 308394 tham số


In [ ]:
import torchvision.transforms as T
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),            # đệm 4 pixel rồi cắt ngẫu nhiên lại 32×32
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
img = T.ToPILImage()(torch.rand(3, 32, 32))           # một ảnh thử
a, b = train_tf(img), train_tf(img)                   # cùng ảnh, hai lần biến đổi ngẫu nhiên khác nhau
print(a.shape, "giống nhau?", torch.equal(a, b))

torch.Size([3, 32, 32]) giống nhau? False


## Bài tập 3: So bốn optimizer trên cùng một bài {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
d = load_digits()
ds = TensorDataset(torch.tensor(d.data / 16, dtype=torch.float32), torch.tensor(d.target))

def chay(make_opt, epochs=10):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
    opt = make_opt(model.parameters())
    loader = DataLoader(ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
    hist = []
    for _ in range(epochs):
        tong = 0.0
        for xb, yb in loader:
            opt.zero_grad(); loss = nn.functional.cross_entropy(model(xb), yb); loss.backward(); opt.step()
            tong += loss.item() * len(xb)
        hist.append(tong / len(ds))
    return hist
# TODO: hist[tên] cho 4 optimizer, vẽ, tot_nhat

In [ ]:
opts = {"SGD": lambda p: torch.optim.SGD(p, lr=0.05),
        "SGD+momentum": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
        "Adam": lambda p: torch.optim.Adam(p, lr=1e-3),
        "AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.01)}
hist = {ten: chay(f) for ten, f in opts.items()}
tot_nhat = min(hist, key=lambda k: hist[k][-1])
fig, ax = plt.subplots(figsize=(6, 3.4))
for ten, h in hist.items():
    ax.plot(range(1, 11), h, marker="o", label=ten)
ax.set(xlabel="epoch", ylabel="loss train", title=f"Thấp nhất sau 10 epoch: {tot_nhat}"); ax.legend()
print({k: round(v[-1], 3) for k, v in hist.items()})

{'SGD': 0.758, 'SGD+momentum': 0.091, 'Adam': 0.388, 'AdamW': 0.389}


## Bài tập 4: Lịch learning rate có warmup {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math, torch
import matplotlib.pyplot as plt
model = torch.nn.Linear(4, 2)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
WARMUP, TOTAL = 100, 1000

def he_so(step):
    # TODO: trả về hệ số nhân với lr tối đa (0..1) ở bước step
    return 1.0

In [ ]:
def he_so(step):
    if step < WARMUP:
        return step / WARMUP
    t = (step - WARMUP) / (TOTAL - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * t))

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, he_so)
lrs = []
for step in range(TOTAL):
    lrs.append(opt.param_groups[0]["lr"])
    opt.step(); sched.step()
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(lrs); ax.set(xlabel="bước", ylabel="learning rate", title="Warmup 100 bước rồi giảm theo cosine")
print(lrs[0], lrs[50], lrs[100], lrs[550], lrs[-1])

0.0 0.00015 0.0003 0.00015 9.138513314410623e-10


## Bài tập 5: CNN nhỏ cho chữ số viết tay {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.datasets import load_digits
d = load_digits()
ds = TensorDataset(torch.tensor(d.images / 16, dtype=torch.float32).unsqueeze(1), torch.tensor(d.target))   # (1797, 1, 8, 8)
torch.manual_seed(0)
train_ds, val_ds = random_split(ds, [1437, 360])
# TODO: cnn, so tham số với MLP, train cả hai, acc_cnn, acc_mlp

In [ ]:
def train_eval(model, epochs=10):
    torch.manual_seed(0)
    loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); nn.functional.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval()
    X = torch.stack([x for x, _ in val_ds]); y = torch.stack([t for _, t in val_ds])
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

torch.manual_seed(0)
cnn = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(32 * 2 * 2, 10))
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
x = torch.zeros(5, 1, 8, 8)
for layer in cnn:
    x = layer(x); print(f"{layer.__class__.__name__:10s} → {tuple(x.shape)}")
acc_cnn, acc_mlp = train_eval(cnn), train_eval(mlp)
print(f"CNN: {sum(p.numel() for p in cnn.parameters())} tham số, val acc {acc_cnn:.3f}")
print(f"MLP: {sum(p.numel() for p in mlp.parameters())} tham số, val acc {acc_mlp:.3f}")

Conv2d     → (5, 16, 8, 8)
ReLU       → (5, 16, 8, 8)
MaxPool2d  → (5, 16, 4, 4)
Conv2d     → (5, 32, 4, 4)
ReLU       → (5, 32, 4, 4)
MaxPool2d  → (5, 32, 2, 2)
Flatten    → (5, 128)
Linear     → (5, 10)
CNN: 6090 tham số, val acc 0.983
MLP: 4810 tham số, val acc 0.947


## Bài tập 6: BatchNorm và Dropout: chế độ train khác chế độ eval {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU(), nn.Dropout(0.5), nn.Linear(8, 2))
x = torch.randn(16, 4)
drop = nn.Dropout(0.5)
ones = torch.ones(10000)
# TODO: giong_train (2 lần ở train mode có giống nhau?), giong_eval, ti_le_0, he_so

In [ ]:
model.train()
giong_train = torch.equal(model(x), model(x))
model.eval()
with torch.no_grad():
    giong_eval = torch.equal(model(x), model(x))
drop.train()
y = drop(ones)
ti_le_0 = (y == 0).float().mean().item()
he_so = y[y > 0][0].item()
print(f"train: giống nhau? {giong_train} | eval: giống nhau? {giong_eval}")
print(f"Dropout(0.5): {ti_le_0:.1%} thành 0, phần còn lại nhân {he_so}")
print("running mean của BatchNorm (đã cập nhật sau các lần chạy train):", model[1].running_mean[:4].round(decimals=3))

train: giống nhau? False | eval: giống nhau? True
Dropout(0.5): 50.4% thành 0, phần còn lại nhân 2.0
running mean của BatchNorm (đã cập nhật sau các lần chạy train): tensor([-0.0330, -0.0490, -0.0560, -0.0990])


## Bài tập 7: Augmentation khi có ít dữ liệu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
d = load_digits()
X = torch.tensor(d.images / 16, dtype=torch.float32).unsqueeze(1); y = torch.tensor(d.target)
X_tr, y_tr, X_va, y_va = X[:300], y[:300], X[1300:], y[1300:]

def dich_ngau_nhien(xb):
    # TODO: dịch mỗi ảnh trong batch 0 hoặc ±1 pixel theo hàng và cột (torch.roll)
    return xb

def train(aug, epochs=100):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(16 * 4 * 4, 10))
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(epochs):
        for i in torch.randperm(300).split(32):
            xb = dich_ngau_nhien(X_tr[i]) if aug else X_tr[i]
            opt.zero_grad(); nn.functional.cross_entropy(model(xb), y_tr[i]).backward(); opt.step()
    with torch.no_grad():
        return (model(X_tr).argmax(1) == y_tr).float().mean().item(), (model(X_va).argmax(1) == y_va).float().mean().item()

In [ ]:
def dich_ngau_nhien(xb):
    out = torch.empty_like(xb)
    for k in range(len(xb)):
        dy, dx = torch.randint(-1, 2, (2,)).tolist()
        out[k] = torch.roll(xb[k], shifts=(dy, dx), dims=(1, 2))
    return out

kq_khong = train(aug=False)
kq_co = train(aug=True)
print(f"không augmentation: train {kq_khong[0]:.3f}, val {kq_khong[1]:.3f}")
print(f"có augmentation   : train {kq_co[0]:.3f}, val {kq_co[1]:.3f}")

không augmentation: train 1.000, val 0.889
có augmentation   : train 0.997, val 0.909
